# Emisiones de CO₂ per cápita — World Bank Open Data

**Indicador:** EN.GHG.CO2.PC.CE.AR5  
**Metodología:** EDGAR/AR5  
**Objetivo:** identificar y diagnosticar los datos históricos de emisiones de CO₂ per cápita por país.


## Parte 1 — Planificación y comprensión

### 1. Clasificación del problema

 Corresponde principalmente a un **problema descriptivo**. Se busca explorar y resumir los datos para identificar qué países presentan mayores emisiones de CO₂. No se busca predecir valores, asignar observaciones a clases previamente definidas ni formar grupos mediante clustering. La finalidad es obtener un diagnóstico que permita priorizar países en negociaciones ambientales.


### 2. CRISP-DM

El trabajo queda principalmente dentro de la fase de Data Understanding, incorporando también tareas de Data Preparation.

Business Understanding: se define el objetivo de identificar países con mayores emisiones para apoyar la priorización de negociaciones ambientales.
Data Understanding: se consulta la API, se inspecciona la estructura de los datos, se identifican países y agregados y se analizan faltantes, cobertura temporal, outliers y duplicados.
Data Preparation: se normaliza el JSON, se separan los agregados de los países y se preparan los datos para su almacenamiento en SQLite.

Las fases que quedan pendientes son Modeling, Evaluation y Deployment. Si el proyecto continuara, el siguiente paso dependería del objetivo final: se podría desarrollar un modelo o análisis específico, evaluar sus resultados y posteriormente desplegar los productos de datos para su utilización operativa.


### 3. Madurez de datos según Gartner

Madurez de datos según Gartner

Puede considerarse que el organismo presenta un nivel de madurez intermedio, ya que cuenta con algunos procesos de adquisición y almacenamiento estructurados, pero todavía depende de fuentes externas y necesita controles adicionales para garantizar la calidad y trazabilidad de los datos antes de utilizarlos para la toma de decisiones.

Dos indicadores concretos que sostienen esta evaluación son:

Dependencia de fuentes externas: el organismo obtiene el indicador directamente de World Bank Open Data, por lo que no controla completamente el proceso de generación ni actualización del dato original.
Necesidad de controles de calidad: antes de utilizar la información es necesario detectar y analizar problemas como valores faltantes, agregados regionales, valores extremos y posibles duplicados.

Esto indica que existe cierta capacidad de gestión y automatización de datos, pero todavía hay oportunidades para fortalecer el gobierno, la validación y el control de calidad de las fuentes utilizadas.


## Parte 2 — Adquisición y trazabilidad

In [ ]:
import requests
import pandas as pd
import numpy as np
import re
import sqlite3
from datetime import date

INDICADOR = "EN.GHG.CO2.PC.CE.AR5"
BASE_URL = f"https://api.worldbank.org/v2/country/all/indicator/{INDICADOR}"
#obtener metadata y total de registros
params = {"format": "json", "per_page": 1}
r = requests.get(BASE_URL, params=params, timeout=30)
r.raise_for_status()
respuesta_1 = r.json()
metadata = respuesta_1[0]
total = int(metadata["total"])
print("Total de registros informado por la API:", total)
print("Páginas:", metadata.get("pages"))


Total de registros informado por la API: 17490
Páginas: 17490


In [ ]:
#traer todos los registros de una vez
params = {"format": "json", "per_page": total}
r = requests.get(BASE_URL, params=params, timeout=60)
r.raise_for_status()
respuesta = r.json()
metadata_final = respuesta[0]
datos = respuesta[1]
print("Registros descargados:", len(datos))
print("Registros esperados:", metadata_final["total"])
print("Coinciden:", len(datos) == int(metadata_final["total"]))


Registros descargados: 17490
Registros esperados: 17490
Coinciden: True


In [ ]:
# Normalización del JSON
df = pd.json_normalize(datos)
print("Columnas normalizadas:")
print(df.columns.tolist())
display(df.head())


Columnas normalizadas:
['countryiso3code', 'date', 'value', 'unit', 'obs_status', 'decimal', 'indicator.id', 'indicator.value', 'country.id', 'country.value']


,countryiso3code,date,value,unit,obs_status,decimal,indicator.id,indicator.value,country.id,country.value
0,AFE,2025,NaN,,,1,EN.GHG.CO2.PC.CE.AR5,Carbon dioxide (CO2) emissions excluding LULUC...,ZH,Africa Eastern and Southern
1,AFE,2024,0.837680,,,1,EN.GHG.CO2.PC.CE.AR5,Carbon dioxide (CO2) emissions excluding LULUC...,ZH,Africa Eastern and Southern
2,AFE,2023,0.838104,,,1,EN.GHG.CO2.PC.CE.AR5,Carbon dioxide (CO2) emissions excluding LULUC...,ZH,Africa Eastern and Southern
3,AFE,2022,0.858132,,,1,EN.GHG.CO2.PC.CE.AR5,Carbon dioxide (CO2) emissions excluding LULUC...,ZH,Africa Eastern and Southern
4,AFE,2021,0.899170,,,1,EN.GHG.CO2.PC.CE.AR5,Carbon dioxide (CO2) emissions excluding LULUC...,ZH,Africa Eastern and Southern


### Campos guardados después de `json_normalize()`

Después de usar `json_normalize()`, los datos que estaban dentro de otros objetos pasan a ser columnas del DataFrame. Por ejemplo, aparecen campos como `indicator.id`, `indicator.value`, `country.id` y `country.value`. Esto permite trabajar con esos datos de una forma más simple.



In [ ]:
AGGREGATE_PATTERN = re.compile(
    r"\b(?:world|income|region|union|euro area|area|ida|ibrd|oecd|"
    r"east asia|south asia|europe|central asia|latin america|caribbean|"
    r"middle east|north africa|arab world|sub-saharan africa|"
    r"africa eastern|africa western|least developed|heavily indebted|"
    r"fragile|small states|pacific island|not classified|demographic dividend)\b",
    flags=re.IGNORECASE)
df["es_agregado"] = df["country.value"].fillna("").str.contains(
    AGGREGATE_PATTERN, regex=True)
df_agregados = df[df["es_agregado"]].copy()
df_paises = df[~df["es_agregado"]].copy()
print("Entidades no detectadas como agregados:", df_paises["country.value"].nunique())
print("Agregados detectados:", df_agregados["country.value"].nunique())
print("\nEjemplos de agregados:")
display(df_agregados[["country.id", "country.value"]].drop_duplicates().head(20))


Entidades no detectadas como agregados: 218
Agregados detectados: 47

Ejemplos de agregados:


,country.id,country.value
0,ZH,Africa Eastern and Southern
66,ZI,Africa Western and Central
132,1A,Arab World
198,S3,Caribbean small states
264,B8,Central Europe and the Baltics
330,V2,Early-demographic dividend
396,Z4,East Asia & Pacific
462,4E,East Asia & Pacific (excluding high income)
528,T4,East Asia & Pacific (IDA & IBRD countries)
594,XC,Euro area


### Criterio de separación

Para separar los países de las entidades agregadas, usé una expresión regular basada en el patrón proporcionado en la consigna. El criterio busca palabras y expresiones características de regiones, grupos de ingresos, uniones y otras agrupaciones, como World, High income, Latin America & Caribbean, OECD, IDA e IBRD.

El resultado fue de 218 entidades no detectadas como agregados y 47 entidades clasificadas como agregados.

Caso límite: el criterio utilizado se basa principalmente en el nombre textual de la entidad. Esto puede generar dudas con territorios y dependencias que aparecen individualmente en la API, como American Samoa, Bermuda, Cayman Islands o Channel Islands. En estos casos, el método permite separarlos de los agregados regionales, pero no resuelve por sí mismo si deben considerarse "países" en sentido político o solamente entidades/territorios. Por ese motivo, para este TP se decidió conservarlos dentro del conjunto de entidades individuales y dejar documentada esta limitación del criterio.


In [ ]:
# Revisión de entidades no detectadas
display(
    df_paises[["country.id", "country.value"]]
    .drop_duplicates()
    .sort_values("country.value")
    .head(40))


,country.id,country.value
3168,AF,Afghanistan
3234,AL,Albania
3300,DZ,Algeria
3366,AS,American Samoa
3432,AD,Andorra
3498,AO,Angola
3564,AG,Antigua and Barbuda
3630,AR,Argentina
3696,AM,Armenia
3762,AW,Aruba


### SQLite

In [ ]:
columnas_sql = [
    "country.id", "country.value", "date", "value",
    "indicator.id", "indicator.value"]
paises_sql = df_paises[columnas_sql].copy()
agregados_sql = df_agregados[columnas_sql].copy()
paises_sql.columns = [
    "country_id", "country_name", "year", "value",
    "indicator_id", "indicator_name"]
agregados_sql.columns = paises_sql.columns
paises_sql["year"] = pd.to_numeric(paises_sql["year"], errors="coerce")
agregados_sql["year"] = pd.to_numeric(agregados_sql["year"], errors="coerce")
paises_sql["value"] = pd.to_numeric(paises_sql["value"], errors="coerce")
agregados_sql["value"] = pd.to_numeric(agregados_sql["value"], errors="coerce")
conn = sqlite3.connect("world_bank_co2.sqlite")
paises_sql.to_sql("paises", conn, if_exists="replace", index=False)
agregados_sql.to_sql("agregados", conn, if_exists="replace", index=False)
#filtrado
consulta_filtro = pd.read_sql(
    "SELECT country_name, year, value "
    "FROM paises WHERE value > 20 "
    "ORDER BY value DESC LIMIT 10;",
    conn)
#agregado
consulta_agregacion = pd.read_sql(
    "SELECT year, AVG(value) AS promedio, COUNT(value) AS observaciones "
    "FROM paises WHERE value IS NOT NULL "
    "GROUP BY year ORDER BY year;",
    conn)
print("Consulta de filtro:")
display(consulta_filtro)
print("Consulta de agregación:")
display(consulta_agregacion.head(10))


Consulta de filtro:


,country_name,year,value
0,Palau,2012,202.865184
1,Palau,1994,178.640429
2,Palau,1970,175.475018
3,Palau,1984,173.067406
4,Palau,1971,169.640276
5,Palau,1972,163.588650
6,Palau,1981,163.272053
7,Palau,1980,162.562138
8,Palau,1979,161.692470
9,Palau,1983,160.472090


Consulta de agregación:


,year,promedio,observaciones
0,1970,5.653731,204
1,1971,5.641221,204
2,1972,5.762565,204
3,1973,6.022694,204
4,1974,5.727841,204
5,1975,5.600825,204
6,1976,5.731045,204
7,1977,5.651545,204
8,1978,5.742648,204
9,1979,5.648156,204


### Ficha de trazabilidad

| Campo | Descripción |
|---|---|
| Origen | World Bank Open Data |
| Fecha de acceso | Se registra automáticamente en la celda siguiente |
| Código | EN.GHG.CO2.PC.CE.AR5 |
| Significado | Indicador de emisiones de CO₂ per cápita |
| Metodología | EDGAR/AR5 |
| Estructura | Registros por entidad y año, con valor del indicador |
| Licencia de uso | Consultar los términos/licencia vigentes de World Bank Open Data |
| Transformaciones | JSON normalizado, agregados separados, tipos convertidos y carga en SQLite |


In [ ]:
print("Fecha de acceso:", date.today().isoformat())
print("Indicador:", INDICADOR)
print("Metodología:", "EDGAR/AR5")


Fecha de acceso: 2026-09-09
Indicador: EN.GHG.CO2.PC.CE.AR5
Metodología: EDGAR/AR5


## Parte 3 — Diagnóstico y tratamiento de calidad

In [ ]:
#Porcentaje de faltantes
total_pais = len(paises_sql)
faltantes = paises_sql["value"].isna().sum()
porcentaje_global = faltantes / total_pais * 100
print(f"Filas de países: {total_pais}")
print(f"Valores faltantes: {faltantes}")
print(f"Porcentaje de faltantes global: {porcentaje_global:.2f}%")


Filas de países: 14388
Valores faltantes: 3168
Porcentaje de faltantes global: 22.02%


In [ ]:
# Ventana temporal y faltantes
años_validos = paises_sql.loc[paises_sql["value"].notna(), "year"]
año_inicio = int(años_validos.min())
año_fin = int(años_validos.max())
ventana = paises_sql[
    paises_sql["year"].between(año_inicio, año_fin)
].copy()
faltantes_ventana = ventana["value"].isna().sum()
porcentaje_ventana = faltantes_ventana / len(ventana) * 100
print(f"Ventana temporal real: {año_inicio} - {año_fin}")
print(f"Faltantes dentro de la ventana: {faltantes_ventana}")
print(f"Porcentaje dentro de la ventana: {porcentaje_ventana:.2f}%")
print(f"Diferencia: {porcentaje_global - porcentaje_ventana:.2f} puntos porcentuales")


Ventana temporal real: 1970 - 2024
Faltantes dentro de la ventana: 770
Porcentaje dentro de la ventana: 6.42%
Diferencia: 15.60 puntos porcentuales


### Interpretación de los faltantes

El porcentaje de faltantes puede ser más alto si se tienen en cuenta todos los años, incluso aquellos en los que todavía no había datos para algunas entidades. Si se analiza solo el período donde el indicador tiene registros, el porcentaje representa mejor la cantidad real de datos faltantes.



In [ ]:
#Faltantes por país
faltantes_por_pais = (
  paises_sql.groupby(
     ["country_id", "country_name"], dropna=False
  )["value"]
    .agg(total="size", con_dato="count"))
faltantes_por_pais["faltantes"] = (
    faltantes_por_pais["total"] - faltantes_por_pais["con_dato"])
faltantes_por_pais["porcentaje_faltante"] = (
    faltantes_por_pais["faltantes"] /
    faltantes_por_pais["total"] * 100)
cien_faltantes = faltantes_por_pais[
faltantes_por_pais["porcentaje_faltante"] == 100]
print("Países con 100% de reportes faltantes:")
display(cien_faltantes.reset_index())


Países con 100% de reportes faltantes:


,country_id,country_name,total,con_dato,faltantes,porcentaje_faltante
0,AD,Andorra,66,0,66,100.0
1,CW,Curacao,66,0,66,100.0
2,IM,Isle of Man,66,0,66,100.0
3,JG,Channel Islands,66,0,66,100.0
4,LI,Liechtenstein,66,0,66,100.0
5,MC,Monaco,66,0,66,100.0
6,ME,Montenegro,66,0,66,100.0
7,MF,St. Martin (French part),66,0,66,100.0
8,PS,West Bank and Gaza,66,0,66,100.0
9,RS,Serbia,66,0,66,100.0


### Tratamiento de faltantes

No se van a completar automáticamente los valores que estén faltando, porque el indicador depende del tiempo y poner un valor que no conocemos podría hacer que los datos sean incorrectos. Por eso, los valores faltantes se dejan como `NaN` y se anotan para tenerlos en cuenta.

Si alguna entidad tiene todos sus valores faltantes, no se utiliza para los cálculos que necesitan datos numéricos. Sin embargo, no se elimina del dataset original, para poder mantener la información completa y saber de dónde vienen los datos.



In [ ]:
#Outliers mediante regla IQR
valores = paises_sql["value"].dropna()
Q1 = valores.quantile(0.25)
Q3 = valores.quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR
outliers = paises_sql[
     paises_sql["value"].notna() &
    ((paises_sql["value"] < limite_inferior) |
        (paises_sql["value"] > limite_superior) )
].copy()
print(f"Q1: {Q1:.4f}")
print(f"Q3: {Q3:.4f}")
print(f"IQR: {IQR:.4f}")
print(f"Límite inferior: {limite_inferior:.4f}")
print(f"Límite superior: {limite_superior:.4f}")
print(f"Cantidad de outliers: {len(outliers)}")
display(
 outliers
  .sort_values("value", ascending=False)
   [["country_id", "country_name", "year", "value"]]
  .head(20))


Q1: 0.3803
Q3: 6.3551
IQR: 5.9747
Límite inferior: -8.5818
Límite superior: 15.3172
Cantidad de outliers: 751


,country_id,country_name,year,value
13081,PW,Palau,2012,202.865184
13099,PW,Palau,1994,178.640429
13123,PW,Palau,1970,175.475018
13109,PW,Palau,1984,173.067406
13122,PW,Palau,1971,169.640276
13121,PW,Palau,1972,163.588650
13112,PW,Palau,1981,163.272053
13113,PW,Palau,1980,162.562138
13114,PW,Palau,1979,161.692470
13110,PW,Palau,1983,160.472090


### Investigación del valor extremo

El valor máximo encontrado corresponde a Palau en 2012, con 202,865 t CO₂e per cápita. El resultado es especialmente llamativo porque se encuentra muy por encima del límite superior calculado mediante IQR. Además, Palau aparece repetidamente entre los valores más altos: también presenta valores superiores a 150 t CO₂e per cápita en varios años anteriores.

Una primera hipótesis es el efecto de la baja población. Al tratarse de un indicador per cápita, las emisiones totales se dividen por una cantidad relativamente pequeña de habitantes. Por lo tanto, un volumen de emisiones que no necesariamente sería extremo al analizarlo en términos absolutos puede producir un valor muy elevado cuando se expresa por habitante.

Otra posible explicación es la estructura energética y económica del país, que puede generar un nivel de emisiones per cápita elevado en relación con su tamaño poblacional. También debe considerarse que el indicador utilizado corresponde a la metodología EDGAR/AR5, por lo que el resultado depende de la metodología y de las estimaciones utilizadas por la fuente.

Por lo tanto, el valor de Palau no debe considerarse automáticamente un error solo por ser un outlier. La hipótesis más razonable es que el resultado esté relacionado con la combinación de un país de población muy pequeña y un nivel determinado de emisiones. Sin información adicional que permita demostrar que el dato es incorrecto, la decisión será conservarlo y marcarlo para revisión contextual, en lugar de eliminarlo.

In [ ]:
#Observación con mayor valor
max_obs = paises_sql.loc[
paises_sql["value"].idxmax(),
["country_id", "country_name", "year", "value"]]
print("Observación de mayor valor:")
display(max_obs.to_frame().T)


Observación de mayor valor:


,country_id,country_name,year,value
13081,PW,Palau,2012,202.865184


In [ ]:
#Duplicados para la país-año
duplicados = paises_sql[
 paises_sql.duplicated(
  subset=["country_id", "year"], keep=False
    )
].sort_values(["country_id", "year"])
print(
    "Cantidad de filas involucradas en duplicados país-año:",
    len(duplicados))
print(
  "Cantidad de combinaciones país-año duplicadas:",
  duplicados.groupby(["country_id", "year"]).ngroups)
if len(duplicados) == 0:
print("Resultado: no existen filas duplicadas para país-año.")
else:
  display(duplicados)


Cantidad de filas involucradas en duplicados país-año: 0
Cantidad de combinaciones país-año duplicadas: 0
Resultado: no existen filas duplicadas para país-año.


### Tratamiento de duplicados

Si no se encuentran registros duplicados, no se hace ningún cambio. En caso de que aparezcan, no se deberían eliminar directamente. Primero habría que revisar si realmente son duplicados o si corresponden a distintos valores, actualizaciones de la fuente o algún problema durante la extracción. Una vez confirmado que son duplicados, se puede dejar un solo registro.



## Parte 4 — Síntesis integradora

El trabajo quedó principalmente en la fase de Data Understanding de CRISP-DM, incorporando también tareas de Data Preparation. Se comprendió la estructura de la información obtenida desde la API del World Bank, se separaron los agregados de las entidades individuales y se analizaron problemas de calidad como valores faltantes, cobertura temporal, outliers y duplicados. También se prepararon los datos para almacenarlos y consultarlos mediante SQLite.

Si el proyecto del organismo climático continuara, el siguiente paso dependería del objetivo definido. Para un análisis descriptivo, se podría avanzar con visualizaciones, indicadores y comparaciones entre países que ayuden a priorizar las negociaciones. Si se quisiera realizar predicciones sobre la evolución futura de las emisiones, sería necesario avanzar hacia Modeling, seleccionar un modelo adecuado y posteriormente evaluar su desempeño antes de utilizarlo operativamente.

La decisión que más dudas generó fue el tratamiento de los outliers, especialmente el caso de Palau. El método IQR identifica valores estadísticamente extremos, pero eso no significa que sean errores. En este caso, el valor de Palau puede estar relacionado con el efecto de expresar las emisiones per cápita en una población pequeña y con características particulares del país. Por eso se decidió conservar el dato y marcarlo para revisión contextual, en lugar de eliminarlo sin evidencia suficiente de que sea incorrecto.

La parte que más dudas me genera es el tratamiento de los **outliers**. Que un dato sea muy diferente al resto no significa necesariamente que esté mal. En este caso, puede haber países con emisiones per cápita muy altas por tener una población pequeña o una economía basada en actividades que generan muchas emisiones. Por eso, no me parece correcto eliminar automáticamente esos datos solo porque el método IQR los marque como extremos. Lo más conveniente sería conservarlos, dejarlos registrados y analizar el contexto del país antes de decidir si realmente hay un problema con el dato.

